# 输出解析器 Output Parser

**语言模型的输出是文本。**

但很多时候，您可能希望获得比纯文本**更结构化的信息**。这就是输出解析器的价值所在。


Output Parser 干两件事：
提前告诉模型：你必须按什么格式输出
模型输出后，把文本自动解析成结构化数据


输出解析器是帮助结构化语言模型响应的类。它们必须实现两种主要方法：

- "get_format_instructions () 获取格式指令"：返回一个包含有关如何格式化语言模型输出的字符串的方法。
- "parse () 解析方法"：接受一个字符串（假设为来自语言模型的响应），并将其解析成某种结构。

然后还有一种可选方法：
- "parse_with_prompt ()使用提示进行解析"：接受一个字符串（假设为来自语言模型的响应）和一个提示（假设为生成此响应的提示），并将其解析成某种结构。在需要重新尝试或修复输出，并且需要从提示中获取信息以执行此操作时，通常会提供提示。

  
区分parse() 和parse_with_prompt() 
parse() 只接收大模型返回的文本，无脑解析；
parse_with_prompt() 多传了原始 prompt + 模型输出文本。
适用场景：当大模型输出格式错乱、缺字段、解析失败时，需要拿着当初的原始提示词做上下文重试、纠错、补全格式，只有这时候才用它；普通正常解析完全用不上。

解析器	用途	        输出结构
Str	  纯文本原样返回	字符串
Json	标准 JSON	字典 dict
Pydantic	强类型实体校验	Pydantic 对象
List	逐条文本清单	列表 list
DateTime	自然语言转时间	datetime 对象
逗号列表	逗号分隔条目	列表 list
Enum	固定枚举选项	枚举值
KeyValue	键值对文本	字典



不管是 List、DateTime、Json、Pydantic：
都有 get_format_instructions 给提示词加格式要求
都有 parse 把模型文本转对应结构化数据
都继承同一个顶层抽象基类，设计思想和 Prompt 模板完全一致：单一职责 + 开闭原则

```
导入对应解析器 + 模型 + 提示模板
初始化 llm 和对应 Parser
获取格式指令 parser.get_format_instructions()
塞进 Prompt
用管道 prompt | llm | parser 组装链路
invoke 调用，直接得到结构化对象 / 字典 / 列表 / 时间
```

## StrOutputParser（纯文本输出，不解析）

最简单，只拿文本，不做结构化。

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatZhipuAI
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"))

# 初始化字符串解析器：原样返回文本
parser = StrOutputParser()

# 管道拼接：模型 -> 解析器
chain = llm | parser
# 调用执行
result = chain.invoke("你好")

# 打印结果：普通字符串
print(result) 

您好！😊很高兴见到您！我是您的智能助手，随时准备为您提供帮助。  

您想聊些什么呢？比如：  
- 日常对话或闲聊  
- 查找信息、解答疑问  
- 创意写作（故事、诗歌等）  
- 学习或工作建议  
- 其他任何需求  

请告诉我您的想法，我会全力为您服务！✨


## JsonOutputParser（输出 JSON → 转字典）
让模型输出 JSON，自动解析为 Python 字典

In [7]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# 1. SON解析器
parser = JsonOutputParser()

# 2. 定义提示模板，注入解析器的格式要求
prompt = PromptTemplate(
    template="给我一个用户信息，严格按JSON格式输出：{format_instructions}\n",
    input_variables=[],
    # 把解析器生成的格式说明，注入模板
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 3. 组装链路：提示词 -> 模型 -> JSON解析
chain = prompt | llm | parser
# 4. 调用并自动解析为字典
result = chain.invoke({})

print(result)
print(type(result))  # <class 'dict'>


{'user_id': 1001, 'username': 'alexchen', 'email': 'alex.chen@example.com', 'full_name': 'Alex Chen', 'created_at': '2023-01-15T08:30:00Z', 'last_login': '2023-10-22T14:45:30Z', 'profile': {'bio': 'Software engineer with passion for AI and cloud technologies.', 'avatar_url': 'https://example.com/avatars/user1001.jpg', 'location': 'San Francisco, CA', 'website': 'https://alexchen.dev'}, 'status': 'active', 'subscription_plan': 'premium', 'preferences': {'language': 'en-US', 'notifications': {'email': True, 'sms': False, 'push': True}}, 'statistics': {'login_count': 127, 'posts_count': 42, 'comments_count': 893}}
<class 'dict'>


## PydanticOutputParser 实体模型解析器（工程最常用）
绑定 Pydantic 模型，自动校验字段、类型，转成实体对象

## ListOutputParser（输出清单 → 转列表）
模型输出编号列表，自动转为 Python list

In [9]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

from langchain_core.prompts import PromptTemplate


parser = CommaSeparatedListOutputParser()

# 2. 提示词+格式约束
prompt = PromptTemplate(
    template="列出3个水果：{format_instructions}",
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser
result = chain.invoke({})

print(result)  # ['Apple', 'Banana', 'Orange']
print(type(result))  # list

['Apple', 'Banana', 'Orange']
<class 'list'>
